# Learning Style Classification Training

This notebook trains a production-ready learning style classifier using `Student_Performance.xlsx`, evaluates it with multiple metrics, visualizes results, and saves reusable Joblib artifacts for the Streamlit app.


In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    f1_score,
    log_loss,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)


In [ ]:
TARGET_COLUMN = "Learning Style"
RANDOM_STATE = 42

dataset_candidates = [
    Path("Student_Performance.xlsx"),
    Path("./Student_Performance.xlsx"),
]

for candidate in dataset_candidates:
    if candidate.exists():
        DATASET_PATH = candidate.resolve()
        break
else:
    raise FileNotFoundError("Student_Performance.xlsx was not found in the project directory.")

MODELS_DIR = Path("models")
MODELS_DIR.mkdir(exist_ok=True)

MODEL_PATH = MODELS_DIR / "learning_style_model.pkl"
PREPROCESSOR_PATH = MODELS_DIR / "preprocessor.pkl"

df = pd.read_excel(DATASET_PATH)
print(f"Dataset loaded from: {DATASET_PATH}")
print(f"Shape: {df.shape}")


In [ ]:
if TARGET_COLUMN not in df.columns:
    raise ValueError(f"Target column '{TARGET_COLUMN}' not found in dataset.")

display(df.head())
display(df.info())
display(df.isna().sum().sort_values(ascending=False).to_frame("missing_values").head(20))
display(df[TARGET_COLUMN].value_counts(dropna=False).to_frame("count"))


In [ ]:
X = df.drop(columns=[TARGET_COLUMN]).copy()
y = df[TARGET_COLUMN].copy()

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = X.select_dtypes(exclude=[np.number]).columns.tolist()

print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("Total features:", X.shape[1])


In [ ]:
numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        (
            "encoder",
            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
        ),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Train shape:", X_train.shape, "->", X_train_processed.shape)
print("Test shape:", X_test.shape, "->", X_test_processed.shape)


In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1,
)

model.fit(X_train_processed, y_train)

y_pred = model.predict(X_test_processed)
y_proba = model.predict_proba(X_test_processed)

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Balanced Accuracy": balanced_accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
    "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
    "F1 Score": f1_score(y_test, y_pred, average="weighted", zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_proba, multi_class="ovr", average="weighted"),
    "MCC": matthews_corrcoef(y_test, y_pred),
    "Cohen Kappa": cohen_kappa_score(y_test, y_pred),
    "Log Loss": log_loss(y_test, y_proba, labels=model.classes_),
}

metrics_df = pd.DataFrame(
    {
        "Metric": list(metrics.keys()),
        "Value": list(metrics.values()),
    }
)
display(metrics_df)


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=model.classes_)

plt.figure(figsize=(8, 6))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=model.classes_,
    yticklabels=model.classes_,
)
plt.title("Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.tight_layout()
plt.show()


In [ ]:
feature_importances = pd.DataFrame(
    {
        "feature": preprocessor.get_feature_names_out(),
        "importance": model.feature_importances_,
    }
).sort_values("importance", ascending=False)

display(feature_importances.head(15))

plt.figure(figsize=(10, 8))
sns.barplot(
    data=feature_importances.head(15),
    x="importance",
    y="feature",
    palette="viridis",
)
plt.title("Top 15 Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()


In [ ]:
# Store metadata directly on the fitted preprocessor so the app can rebuild forms and validations.
preprocessor.feature_columns_ = X.columns.tolist()
preprocessor.numeric_features_ = numeric_features
preprocessor.categorical_features_ = categorical_features
preprocessor.target_column_ = TARGET_COLUMN
preprocessor.class_names_ = model.classes_.tolist()
preprocessor.train_shape_ = X_train.shape
preprocessor.test_shape_ = X_test.shape
preprocessor.random_state_ = RANDOM_STATE
preprocessor.model_name_ = type(model).__name__

joblib.dump(model, MODEL_PATH)
joblib.dump(preprocessor, PREPROCESSOR_PATH)

print(f"Saved model to: {MODEL_PATH.resolve()}")
print(f"Saved preprocessor to: {PREPROCESSOR_PATH.resolve()}")
